## Fitting Lyα Radiative Transfer Models with zELDA

In this notebook, we fit homogeneous thin expanding shell radiative transfer models using grids and MCMC implementation provided in the zELDA package (Gurung-Lopez et al. 2021). Note that zELDA has very particular dependencies, so may require its own dedicated venv to run this notebook.

In [ ]:
import numpy as np
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import Lya_zelda as Lya
import gc


from tangelo import io as tgio

In [ ]:
# Where to get the spectra from
SPEC_SOURCE    = 'R21' # 'APER' or 'R21'
SPEC_TYPE      = 'weight_skysub' # type of spectrum, e.g. '1fwhm' for 'aper' or 'noweight_skysub' for 'R21'

# Where to look for spectra
spec_dir = tgio.get_r21_spectra_dir if SPEC_SOURCE == 'R21' else tgio.get_aper_spectra_dir
filepattern = lambda iden: f"{'spec_' if SPEC_SOURCE == 'R21' else ''}{iden}_{SPEC_TYPE}.fits"

# Using this we define the function used to load spectra
spec_meth = lambda iden, cluster: tgio.load_spec(cluster, iden, '', spec_type=SPEC_TYPE, spec_source=SPEC_SOURCE)

In [ ]:
# Load the megatable with LAEs
megatab_file = f"./megatables/lae_megatab_flagged_cpts_refit_zeldamcmc_{SPEC_TYPE}.fits" # CHANGE IF RESTART FROM SCRATCH
megatab = aptb.Table(fits.open(megatab_file)[1].data)
print(f"Loaded {len(megatab)} LAEs from {megatab_file}")

# Remove rows with no significant Lyman alpha
megatab = megatab[(megatab['SNRR'] > 5.0) | (megatab['SNRB'] > 5.0)]
megatab.sort(keys='z')
print(f"Filtered down to {len(megatab)} LAEs")

In [ ]:
# Load the Lyman alpha model grids

# Function to isolate the grid loading process
def load_grid():
    """ Load the Lyman alpha radiative transfer grid with memory monitoring"""
    grids_location = '../lya_rt_grids/Grids'
    Lya.funcs.Data_location = grids_location
    
    Geometry = 'Thin_Shell_Cont'
    # Clear any previous loaded grids from memory
    if 'LyaRT_Grid' in globals():
        del LyaRT_Grid
    gc.collect()
    
    # Load the grid with memory monitoring
    return Lya.load_Grid_Line(Geometry, MODE='FULL')

LyaRT_Grid = load_grid()

In [ ]:
def opfunc(x, a, b):
    """ Helper function that can be used to re-normalise a model by fitting it to data """
    return a * x + b

In [ ]:
# Define physical parameters to be used in the script
from tangelo import constants as tgconst

# Get c from constants
c = tgconst.c
# Dictionary of wavelengths
wavedict = tgconst.wavedict

In [ ]:
def is_absorber(tab, wavedict, sig = 3.0, n = 1):
    tv = np.zeros(len(tab)).astype(int)
    for line in wavedict:
        if line == 'LYALPHA':
            continue
        else:
            try:
                tv += ((tab[f'SNR_{line}'] < -sig) 
                       * (tab[f'SNR_{line}'] > -1000.)
                       * (tab[f'FLAG_{line}'] != 'c')).astype(int)
            except:
                continue
    return tv >= n

def is_emitter(tab, wavedict, sig = 3.0, n = 1):
    tv = np.zeros(len(tab)).astype(int)
    for line in wavedict:
        if line == 'LYALPHA':
            continue
        else:
            try:
                tv += ((tab[f'SNR_{line}'] > sig) 
                       * (tab[f'SNR_{line}'] < 1000.)
                       * (tab[f'FLAG_{line}'] != 'c')).astype(int)
            except:
                continue
    return tv >= n

In [ ]:
# Print out the parameters and their values in the grid

print( 'The expansion velocity [km/s] is evaluated in : ')

print( LyaRT_Grid['V_Arr'    ] )

print( 'The logarithmic of the HI column density [cm**-2] is evaluated in : ')

print( LyaRT_Grid['logNH_Arr'] )

print( 'The logarithmic of the dust optical depth is evaluated in : ')

print( LyaRT_Grid['logta_Arr'] )

print( 'The logarithmic of the intrinsic equivalent width [A] is evaluated in : ')
print( LyaRT_Grid['logEW_Arr'] )

print( 'The logarithmic of the intrinsic line width [A] is evaluated in : ')
print( LyaRT_Grid['Wi_Arr'] )

In [ ]:
# Sort the megatable by SNRR
megatab.sort('SNRR', reverse=True)

In [ ]:
# USE FULL MCMC FITTING WITH ZELDA
from tangelo import spectroscopy as tgspec

# Generate table columns
if 'Z_ZELDA' not in megatab.colnames:
    megatab['RCHSQ_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['Z_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['Z_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['Z_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['Z_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan
    megatab['VEXP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['VEXP_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['VEXP_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['VEXP_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGN_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGN_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGN_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGN_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan
    megatab['TDUST_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['TDUST_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['TDUST_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['TDUST_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGIEW_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGIEW_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGIEW_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['LOGIEW_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan
    megatab['WINT_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['WINT_ERRM_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['WINT_ERRP_ZELDA'] = np.zeros(len(megatab)) * np.nan
    megatab['WINT_ZELDA_ML'] = np.zeros(len(megatab)) * np.nan

# What geometry to use here?
Geometry = 'Thin_Shell_Cont'

tests = [
    "7681",
    "00080",
    "5399",
    "0001292",
    "3328",
    "5624",
    "2159",
    "10503",
    "7737",
    "0004770",
    "375",
    "3031",
    "8683",
    "653",
    "484",
    "632",
    "470",
    "0005393",
    "305",
    "573",
    "3035",
    "0001050",
    "869",
    "8253",
    "7858",
    "6228"
]

megatab_writename = f"./megatables/lae_megatab_flagged_cpts_refit_zeldamcmc_{SPEC_TYPE}.fits"

import signal

# 1. Define a custom exception for the timeout
class TimeoutException(Exception):
    pass

# 2. Define the signal handler
def signal_handler(signum, frame):
    raise TimeoutException("Iteration timed out")

# 3. Register the signal handler
signal.signal(signal.SIGALRM, signal_handler)

# 4. Implement the loop with a try...except block
timeout_seconds = 180 # Set your desired timeout

for i, row in enumerate(megatab):
#     if row['iden'] not in tests:
#         continue
    if row['Z_ZELDA'] > 0:
        continue
    signal.alarm(timeout_seconds) # Set the alarm for the current iteration
    print(f"Processing source {row['iden']} from cluster {row['CLUSTER']} with zELDA")

    try:
        # Generate spectrum
        print(f"Generating spectrum...")
        w_Arr, f_Arr, s_Arr = tgspec.get_line_spec(row, line='LYALPHA', width=5.0, rest=True, spec_source=SPEC_SOURCE,
                                                   spec_type=SPEC_TYPE)

        # Normalise the spectrum so that total Lyman alpha flux = 1
        f_Arr_n = f_Arr / np.nansum([row['FLUXR'], row['FLUXB']])
        s_Arr_n = s_Arr / np.nansum([row['FLUXR'], row['FLUXB']])

        # Get peak SNR
        peak = row['AMPR']
        noise = row['AMPR_ERR']
        PNR_t  = peak / noise # Signal to noise ratio of the maximum of the line.

        # Get LSF FWHM and pixel scale in Angstroms
        FWHM_t = tgspec.muse_lsf_fwhm_poly(row['LPEAKR'])  # Full width half maximum diluting the line. Mimics finite resolution. [A]
        PIX_t  = 1.25  # Wavelength binning of the line. [A]
    
        # MCMC initialization
        N_walkers = 100 # Number of walkers
        N_burn    = 100 # Number of steps to burn-in
        N_steps   = 150 # Number of steps to run after burning-in
        MODE      = 'DNN' #use NN to initialize the walkers. Can change to PSO if not working.
        print(f"Initializing MCMC...")
        log_V_in, log_N_in, log_t_in, log_E_in, \
            W_in, z_in, Best = Lya.MCMC_get_region_6D(MODE, w_Arr, f_Arr_n, 
                                                        s_Arr_n, FWHM_t, PIX_t, 
                                                        LyaRT_Grid, Geometry) #gets regions to spawn walkers. 
                                                                                #(NOT PRIORS) You can manually set also.

        print(f"Running MCMC...")
        # Perform MCMC
        sampler = Lya.MCMC_Analysis_sampler_5(w_Arr, f_Arr_n, s_Arr_n, FWHM_t, 
                                              N_walkers, N_burn, N_steps, Geometry,
                                              LyaRT_Grid, z_in = z_in, log_V_in = log_V_in,
                                              log_N_in = log_N_in, log_t_in = log_t_in, 
                                              log_E_in = log_E_in, W_in = W_in)

        print(f"Done. Processing results...")

    
        # Get MCMC results
        Q_Arr = [2.5, 50, 97.5] # You can add more percentiles here, like 95
        perc_matrix_sol, flat_samples = Lya.get_solutions_from_sampler(sampler, N_walkers, N_burn, N_steps, Q_Arr)

        # redshift:
        z_16     =     perc_matrix_sol[ 3 , 0 ] # corresponds to Q_Arr[0]
        z_50     =     perc_matrix_sol[ 3 , 1 ] # corresponds to Q_Arr[1]
        z_84     =     perc_matrix_sol[ 3 , 2 ] # corresponds to Q_Arr[2]
        # Expansion velocity.
        V_16     = 10**perc_matrix_sol[ 0 , 0 ]
        V_50     = 10**perc_matrix_sol[ 0 , 1 ]
        V_84     = 10**perc_matrix_sol[ 0 , 2 ]
        # dust optical depth.
        t_16     = 10**perc_matrix_sol[ 2 , 0 ]
        t_50     = 10**perc_matrix_sol[ 2 , 1 ]
        t_84     = 10**perc_matrix_sol[ 2 , 2 ]
        # Intrinsic width.
        W_16     =     perc_matrix_sol[ 5 , 0 ]
        W_50     =     perc_matrix_sol[ 5 , 1 ]
        W_84     =     perc_matrix_sol[ 5 , 2 ]
        # Logarithmic of the intrinsic equivalent width.
        log_E_16 =     perc_matrix_sol[ 4 , 0 ]
        log_E_50 =     perc_matrix_sol[ 4 , 1 ]
        log_E_84 =     perc_matrix_sol[ 4 , 2 ]
        # Logarithmic of the HI column density.
        log_N_16 =     perc_matrix_sol[ 1 , 0 ]
        log_N_50 =     perc_matrix_sol[ 1 , 1 ]
        log_N_84 =     perc_matrix_sol[ 1 , 2 ]

        if t_16 > 0.1:
            print(f"Genuinely high-dust fit over 95% confidence interval!")
    
        # Now generate the model spectra, and compare with the fitted profiles. We will use ML models here,
        # found using the maximum log likelihood
        max_lnprob_idx = np.argmax(sampler.flatlnprobability) # Get the index of the maximum log probability
        best_params_ml = sampler.flatchain[max_lnprob_idx] # Extract the best parameters
        max_lnprob     = sampler.flatlnprobability[max_lnprob_idx] # The corresponding maximum log probability value
        PNR     = 100000.
        F_t      = np.nansum([row['FLUXR'], row['FLUXB']])
        z_sol    = best_params_ml[3] #ML z
        V_sol    = 10**best_params_ml[0] # Expansion velocity km/s
        logN_sol = best_params_ml[1] # log of HI column density cm**-2
        t_sol    = 10**best_params_ml[2] # dust optical depth
        logE_sol = best_params_ml[4] # log intrinsic EW [A]
        W_sol    = best_params_ml[5] # intrinsic width [A]
        w_pix_One_Arr , f_pix_One_Arr , _  = Lya.Generate_a_real_line(
            z_sol , V_sol, logN_sol, t_sol, F_t, logE_sol, W_sol, PNR, FWHM_t, PIX_t / 10, LyaRT_Grid, Geometry
        )
        
        # Now match the simulated line to the data
        # f_pix_One_Arr *= np.nanmax([_row['AMPR'], _row['AMPB']]) / (np.max(f_pix_One_Arr) - f_pix_One_Arr[0])
        f_pix_One_Arr_res = np.interp(w_Arr, w_pix_One_Arr, f_pix_One_Arr)
        popt, pcov = curve_fit(opfunc, f_pix_One_Arr_res[f_pix_One_Arr_res < np.inf], 
                               f_Arr[f_Arr < np.inf], p0=[1,0])
        f_pix_refitted = opfunc(f_pix_One_Arr_res, *popt)
        f_pix_refitted_hires = opfunc(f_pix_One_Arr, *popt)

        # Calculate the reduched chi square over the entire range used to fit (only fair)
        chsq = np.nansum(np.square((f_pix_refitted - f_Arr) / s_Arr)) 
        rchsq = chsq / (np.size(f_Arr) - 6)
        print(f"model reduced chi squared = {rchsq}")
        # rchsqs.append(rchsq)
        # print(f"Model reduced chi squared: {rchsq} (vs {mod_rchsq} for analytic fit).")

        # Plotting
        fig, ax = plt.subplots(figsize=(15,5), facecolor='w')
        ax.step( w_Arr     , f_Arr     , label='Target' , where='mid', color='slateblue')
        ax.fill_between(w_Arr, f_Arr - s_Arr, f_Arr + s_Arr, edgecolor='none', color='slateblue', alpha=0.2, step='mid')
        ax.plot( w_pix_One_Arr , f_pix_refitted_hires , label='1 iter' , color='fuchsia', drawstyle='steps-mid')
        ax.legend(loc=0)
        ax.set_xlabel('wavelength[A]' , size=15)
        ax.set_ylabel('Flux density [erg/s/cm/cm/A]' , size=15 )
        ax.set_xlim(w_Arr[0] - 5., w_Arr[-1] + 5.)
        ax.axvline(1215.67 * (z_sol + 1), linestyle='--', alpha=0.5, color='black')
        ax.axhline(0., linestyle=':', alpha=0.3)
        plot_dir = tgio.get_plot_dir(row['CLUSTER'], row['iden'])
        plot_path = plot_dir / f"LYALPHA_rt_fit.png"
        fig.savefig(plot_path, dpi=300)
        plt.show()#now generate the model spectra, and compare with the fitted profiles. If poor agreement, reject result:
        plt.close()

        # #warning if the fit doesn't look very good:
        # if rchsq > np.nanmax([5.0, mod_rchsq + 0.1]) or np.abs(true_max - sim_max) > np.nanmax([3 * true_maxerr, 1.25]):
        #     print(f"Model produced by zELDA in poor agreement with fitted model. Should be flagged/omitted"
        #             "in any future statistics!")

        # Insert results into the megatable
        # Reduced chi-squared
        megatab['RCHSQ_ZELDA'][i] = rchsq
        # Redshift
        megatab['Z_ZELDA'][i] = z_50
        megatab['Z_ERRM_ZELDA'][i] = np.abs(z_50 - z_16)
        megatab['Z_ERRP_ZELDA'][i] = np.abs(z_84 - z_50)
        megatab['Z_ZELDA_ML'][i] = z_sol
        # Expansion velocity
        megatab['VEXP_ZELDA'][i] = V_50
        megatab['VEXP_ERRM_ZELDA'][i] = np.abs(V_50 - V_16)
        megatab['VEXP_ERRP_ZELDA'][i] = np.abs(V_84 - V_50)
        megatab['VEXP_ZELDA_ML'][i] = V_sol
        # Logarithmic of HI column density
        megatab['LOGN_ZELDA'][i] = log_N_50
        megatab['LOGN_ERRM_ZELDA'][i] = np.abs(log_N_50 - log_N_16)
        megatab['LOGN_ERRP_ZELDA'][i] = np.abs(log_N_84 - log_N_50)
        megatab['LOGN_ZELDA_ML'][i] = logN_sol
        # Dust optical depth
        megatab['TDUST_ZELDA'][i] = t_50
        megatab['TDUST_ERRM_ZELDA'][i] = np.abs(t_50 - t_16)
        megatab['TDUST_ERRP_ZELDA'][i] = np.abs(t_84 - t_50)
        megatab['TDUST_ZELDA_ML'][i] = t_sol
        # Logarithmic of intrinsic equivalent width
        megatab['LOGIEW_ZELDA'][i] = log_E_50
        megatab['LOGIEW_ERRM_ZELDA'][i] = np.abs(log_E_50 - log_E_16)
        megatab['LOGIEW_ERRP_ZELDA'][i] = np.abs(log_E_84 - log_E_50)
        megatab['LOGIEW_ZELDA_ML'][i] = logE_sol
        # Intrinsic width
        megatab['WINT_ZELDA'][i] = W_50
        megatab['WINT_ERRM_ZELDA'][i] = np.abs(W_50 - W_16)
        megatab['WINT_ERRP_ZELDA'][i] = np.abs(W_84 - W_50)
        megatab['WINT_ZELDA_ML'][i] = W_sol
        # Save the megatab in case of a crash
        megatab.write(megatab_writename, overwrite=True)
        print("Done.")
    except TimeoutException as e:
        print(f"Timed out. Skipping to next iteration.")
        continue # Skip the rest of the current iteration
    except Exception as e:
        print(e)
        print("Cannot finish processing; moving to next source")
        continue
    print( 'The measured redshift                                                     is' , z_sol    )
    print( 'The measured expasion velocity                                            is' , V_sol    )
    print( 'The measured logarithm of the HI column density                           is' , logN_sol )
    print( 'The measured dust optical depth                                           is' , t_sol    )
    print( 'The measured logarithm of the intrinsic equivalent width                  is' , logE_sol )
    print( 'The measured intrinsic width                                              is' , W_sol    )

In [ ]:
# Finally, save the megatable with all the results
megatab.write(megatab_writename, overwrite=True)